In [1]:
from torch.utils.data import DataLoader
import dataloader_direct as dataloader
from direct_models import *
import torch
import numpy as np
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
from torch import nn
from torch.nn import functional as F
import os
import shutil
import json
import matplotlib.pyplot as plt
import time

from torch.utils.tensorboard import SummaryWriter

import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

In [2]:
config = json.load(open('config.json'))

directory = config['directory']


## Set-up tensorboard writer
writer = SummaryWriter(log_dir=directory)


## Prepare dataset
train_dataset = dataloader.direct_dataset(config['target_array'], config['spectrum_h5'], 
                                           i_start_ds = config['i_start_ds'], i_end_ds = config['i_end_ds'],
                                           mean_normalization_json = config['mean_normalization_json'],
                                           std_normalization_json = config['std_normalization_json'],
                                           mean_normalization_npy = config['mean_normalization_npy'],
                                           std_normalization_npy = config['std_normalization_npy'],
                                           normalize_spectra = config['normalize_spectra'],
                                           normalize_target = config['normalize_target'])

val_dataset = dataloader.direct_dataset(config['target_array'], config['spectrum_h5'], 
                                           i_start_ds = config['i_start_val_ds'], i_end_ds = config['i_end_val_ds'],
                                           mean_normalization_json = config['mean_normalization_json'],
                                           std_normalization_json = config['std_normalization_json'],
                                           mean_normalization_npy = config['mean_normalization_npy'],
                                           std_normalization_npy = config['std_normalization_npy'],
                                           normalize_spectra = config['normalize_spectra'],
                                           normalize_target = config['normalize_target'])

use_subsampler = config['use_subsampler']
with_wind=config['with_wind']

In [4]:
## Training parameters
nb_epochs = config['nb_epochs'] 
batch_size = config['batch_size']
val_batch_size = config['val_batch_size']
lr_initial = config['learning_rate_ini'] 

model_name = config['model_name']
n_channels = config['n_channels']

use_scheduler = config['scheduler']
if use_scheduler == 'step':
    lr_sched_step = config['lr_sched_step'] 
    lr_sched_gamma = config['lr_sched_gamma'] 
elif use_scheduler == 'plateau':
    lr_factor = config['lr_factor']
    lr_patience = config['lr_patience']
    lr_threshold = config['lr_threshold']
    lr_min = config['lr_min']
    
checkpoint_name = directory+config['checkpoint_name']

ts_size_per_epoch = 100000#config['ts_size_per_epoch']
num_workers_loader = config['num_workers_loader']


    
## Set-up model, optimizer and scheduler
device = config['device']
model = locals()[model_name](n_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr_initial)
if use_scheduler=='step':
    scheduler = StepLR(optimizer, step_size=lr_sched_step, gamma=lr_sched_gamma)
elif use_scheduler=='plateau':
    scheduler = ReduceLROnPlateau(optimizer, factor = lr_factor, patience = lr_patience, threshold = lr_threshold, min_lr = lr_min)
stop_decreasing_after = config['stop_decreasing_after']

init = config['init']
if init=='xavier_normal':
    def weights_init(m):
        if isinstance(m, nn.Conv1d):
            torch.nn.init.xavier_normal_(m.weight)
            torch.nn.init.zeros_(m.bias)
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_normal_(m.weight)
            torch.nn.init.zeros_(m.bias)
    model.apply(weights_init)
    
    
## Load the checkpoint if it exists
bc = 0
nb_epochs_finished = 0
N = train_dataset.spectrum_W.shape[0]
epoch_loss_previous = 0
try:
    checkpoint = torch.load(checkpoint_name)
    nb_epochs_finished = checkpoint['nb_epochs_finished']
    bc = checkpoint['nb_batches_finished']
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    if (use_scheduler=='step') & (nb_epochs_finished<=stop_decreasing_after):
        scheduler.load_state_dict(checkpoint['scheduler_state'])
    print(f'Checkpoint loaded with {nb_epochs_finished} epochs finished.')
except FileNotFoundError:
    print('Starting from scratch.')
except Exception as e:
    print('Error when loading the checkpoint: ', e)
    exit(1)

Starting from scratch.


In [5]:
    
## Perform the training
# The validation dataloader is defined before training
val_dataloader = DataLoader(val_dataset, num_workers=num_workers_loader, batch_size=val_batch_size, pin_memory=True)

# If we don't use the subsampler, the dataloader is defined once and for all  
if not(use_subsampler):
    train_dataloader = DataLoader(train_dataset, num_workers=num_workers_loader, batch_size=batch_size,pin_memory=True)
    

In [6]:
for epoch in range(nb_epochs_finished,nb_epochs):
    epochloss = 0
    
    if use_subsampler:
        inds = torch.arange((ts_size_per_epoch*epoch)%N, min(N,(ts_size_per_epoch*epoch)%N + ts_size_per_epoch))
        train_dataloader = DataLoader(train_dataset, num_workers=num_workers_loader, batch_size=batch_size, sampler = torch.utils.data.SubsetRandomSampler(inds), pin_memory=True)
        print(len(train_dataloader))
        
    for (input_data, target) in iter(train_dataloader):
        input_data = input_data.to(device)
        target = target.to(device)
        output = model(input_data)
        loss = 0.5 * ((output[:,0,:] - target).pow(2).sum())/ output.size(0)
        
        
        epochloss+=loss*output.size(0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if bc%5==0:
            print(bc,loss)
        bc+=1
       
        
    epoch_loss_previous = epochloss/ts_size_per_epoch
    

400
0 tensor(27.7218, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
5 tensor(31.1213, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
10 tensor(11.8466, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
15 tensor(39.1678, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
20 tensor(16.4830, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
25 tensor(18.0396, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
30 tensor(423.4893, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
35 tensor(8.4796, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
40 tensor(84.7731, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
45 tensor(22.7954, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
50 tensor(5.5537, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
55 tensor(424.3836, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
60 tensor(285.2601, device

[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warn

400 tensor(33.3576, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
405 tensor(8.0310, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
410 tensor(14.4967, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
415 tensor(9.8129, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
420 tensor(6.1438, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
425 tensor(5.9308, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
430 tensor(9.5760, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
435 tensor(54.7697, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
440 tensor(5.1828, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
445 tensor(15.1757, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
450 tensor(8.7691, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
455 tensor(5.5014, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
460 tensor(13.9934, devi

[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)


685 tensor(5.1761, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
690 tensor(5.2264, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)
695 tensor(2.8729, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)


KeyboardInterrupt: 

In [11]:
0.5 * ((output[:,0,:] - target).pow(2).sum())/ output.size(0)

tensor(22.8222, device='cuda:0', dtype=torch.float64, grad_fn=<DivBackward0>)

In [6]:
target.shape

torch.Size([250, 12])

In [11]:
model

Direct_w_Resblock(
  (conv1): Conv1d(2, 20, kernel_size=(7,), stride=(1,))
  (resblock1): ResBlock(
    (conv1): Conv1d(20, 20, kernel_size=(7,), stride=(1,), padding=(3,), padding_mode=replicate)
    (bn1): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(20, 20, kernel_size=(7,), stride=(1,), padding=(3,), padding_mode=replicate)
    (bn2): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (avgpool1): AvgPool1d(kernel_size=(3,), stride=(3,), padding=(0,))
  (resblock2): ResBlock(
    (conv1): Conv1d(20, 20, kernel_size=(7,), stride=(1,), padding=(3,), padding_mode=replicate)
    (bn1): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(20, 20, kernel_size=(7,), stride=(1,), padding=(3,), padding_mode=replicate)
    (bn2): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (avgpool2): AvgPool1d(kernel_size=(3,), stri